# Ingestión de Datos desde un API

Este notebook realiza la ingesta de datos desde un API, los almacena en una base de datos SQLite, genera un archivo Excel con una muestra de los datos mediante Pandas y finalmente crea un reporte de auditoría en texto.

In [ ]:
import os
import requests
import sqlite3
import pandas as pd
from datetime import datetime

# Definir directorios base (en caso de ejecutar desde la raíz u otra carpeta)
BASE_DIR = os.getcwd()
SRC_DIR = os.path.join(BASE_DIR, 'src')
DB_DIR = os.path.join(SRC_DIR, 'db')
XLSX_DIR = os.path.join(SRC_DIR, 'xlsx')
AUDIT_DIR = os.path.join(SRC_DIR, 'static', 'auditoria')

# Asegurar que existan los directorios
for d in [DB_DIR, XLSX_DIR, AUDIT_DIR]:
    os.makedirs(d, exist_ok=True)

DB_PATH = os.path.join(DB_DIR, 'ingestion.db')
XLSX_PATH = os.path.join(XLSX_DIR, 'ingestion.xlsx')
AUDIT_PATH = os.path.join(AUDIT_DIR, 'ingestion.txt')

API_URL = "https://jsonplaceholder.typicode.com/users"
print("Configuración y directorios listos.")

## 1. Extracción de Datos desde el API

In [ ]:
print(f"[{datetime.now()}] Extrayendo datos de: {API_URL}")
response = requests.get(API_URL)
response.raise_for_status()
api_data = response.json()
print(f"Se extrajeron {len(api_data)} registros exitosamente.")
api_data[:2] # Mostrar los primeros 2 registros

## 2. Almacenamiento en SQLite

In [ ]:
print(f"[{datetime.now()}] Guardando datos en SQLite: {DB_PATH}")
conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

# Crear tabla
cursor.execute('''
    CREATE TABLE IF NOT EXISTS users (
        id INTEGER PRIMARY KEY,
        name TEXT,
        username TEXT,
        email TEXT,
        phone TEXT,
        website TEXT
    )
''')

# Insertar datos
for user in api_data:
    cursor.execute('''
        INSERT OR REPLACE INTO users (id, name, username, email, phone, website)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (user['id'], user['name'], user['username'], user['email'], user['phone'], user['website']))

conn.commit()
conn.close()
print("Datos guardados en la base de datos correctamente.")

## 3. Generación de Evidencias (Muestra en Archivo Excel con Pandas)

In [ ]:
print(f"[{datetime.now()}] Generando muestra en XLSX: {XLSX_PATH}")
conn = sqlite3.connect(DB_PATH)
db_df = pd.read_sql_query("SELECT * FROM users", conn)
conn.close()

# Seleccionar muestra y exportar
sample_df = db_df.head(10)
sample_df.to_excel(XLSX_PATH, index=False)
print("Archivo excel generado.")
sample_df # Mostrar la muestra en el notebook

## 4. Archivo de Auditoría (.txt)

In [ ]:
print(f"[{datetime.now()}] Generando auditoría: {AUDIT_PATH}")
api_count = len(api_data)
db_count = len(db_df)

with open(AUDIT_PATH, 'w', encoding='utf-8') as f:
    f.write("--- REPORTE DE AUDITORÍA ---\n")
    f.write(f"Fecha: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write(f"API: {API_URL}\n\n")
    f.write("1. Conteo de Registros:\n")
    f.write(f" - API: {api_count}\n")
    f.write(f" - BD: {db_count}\n\n")
    
    if api_count == db_count:
        f.write("ESTADO: ÉXITO. Los registros concuerdan.\n\n")
    else:
        f.write("ESTADO: ADVERTENCIA. Diferencia en registros.\n\n")
    
    f.write("2. Verificación de Integridad:\n")
    f.write(" - Estructura de tabla verificada.\n")
    f.write(" - Archivo XLSX generado exitosamente.\n")

print("Auditoría completada. Todo el proceso ha finalizado correctamente.")